# YOLO Auto-Annotator

This notebook trains a YOLO model on existing YOLO-format dataset and then auto-annotates a folder of new, unlabeled images by generating YOLO-format label `.txt` files.

### What this notebook does
1. Installs dependencies (Ultralytics YOLO, OpenCV, etc.).
2. Runs predictions over your **new images**.
3. Saves predicted boxes as YOLO-format labels (with confidence), mirroring YOLO's `labels/` structure.

> **Note**: Auto-annotations are only as good as the model. Always spot-check and correct labels before training!

## 1) Configure paths and options


In [2]:
# %%capture
!python.exe -m pip install --upgrade pip
!pip -q install ultralytics opencv-python tqdm streamlit pillow pyyaml labelimg
import os, sys, glob, shutil, math, json
from pathlib import Path
from typing import List, Optional
import shutil
import numpy as np
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO
from datetime import datetime
from PIL import Image, ImageDraw, ImageFont
import streamlit as st
import yaml
print('Environment ready.')

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 12.1 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Environment ready.


In [3]:
def resolve(p: Optional[str], base: Path) -> Optional[Path]:
    if p is None:
        return None
    # Expand ~ and environment variables, allow forward slashes on Windows
    p = os.path.expandvars(os.path.expanduser(p))
    q = Path(p)
    return (base / q).resolve() if not q.is_absolute() else q.resolve()

# === REQUIRED: Paths ===
BASE_DIR = Path.cwd().resolve()
print("BASE_DIR:", BASE_DIR)
DATA_YAML = resolve("data.yaml", BASE_DIR)
print("DATA_YAML:", DATA_YAML)
LABELED_DATASET_ROOT = resolve("dataset", BASE_DIR)
print("LABELED_DATASET_ROOT:", LABELED_DATASET_ROOT)
NEW_IMAGES_DIR = resolve("new_images", BASE_DIR)
print("NEW_IMAGES_DIR:", NEW_IMAGES_DIR)
LABELIMG_CMD = "labelImg"

# === Model selection ===
# If you have a fine-tuned model already, put its path here, e.g. "runs/detect/train/weights/best.pt"
CUSTOM_MODEL_PATH: Optional[str] = "runs/detect/first/weights/best.pt"  # or set to a .pt file
PRETRAINED_MODEL_NAME = "yolov8n.pt"    # used if CUSTOM_MODEL_PATH is None

# === Training options ===
TRAIN_MODE = "skip"  # one of: 'skip', 'train', 'resume'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = 0  # set to 'cpu' to force CPU

# === Prediction / auto-annotation options ===
CONF_THRES = 0.25     # confidence threshold for saving detections
IOU_THRES = 0.7       # NMS IOU
MAX_DET = 300         # max detections per image
SAVE_CONF = True      # include confidence values in the label files
SAVE_VIS = True       # save visualization images

# === Output structure ===
OUTPUT_ROOT = str(Path(NEW_IMAGES_DIR).absolute())
OUTPUT_LABELS_DIR = str(Path(OUTPUT_ROOT) / "labels")
OUTPUT_VIS_DIR = str(Path(OUTPUT_ROOT) / "pred_vis")
os.makedirs(OUTPUT_LABELS_DIR, exist_ok=True)
if SAVE_VIS:
    os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)

img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

print("Configured.")

BASE_DIR: C:\Users\omer_\PycharmProjects\YOLO
DATA_YAML: C:\Users\omer_\PycharmProjects\YOLO\data.yaml
LABELED_DATASET_ROOT: C:\Users\omer_\PycharmProjects\YOLO\dataset
NEW_IMAGES_DIR: C:\Users\omer_\PycharmProjects\YOLO\new_images
Configured.


## 2) Utility functions (YOLO-format helpers & drawing)

In [4]:
def xyxy_to_xywh_norm(box_xyxy, img_w, img_h):
    # box_xyxy: [x1, y1, x2, y2] absolute pixels
    x1, y1, x2, y2 = box_xyxy
    w = max(0.0, x2 - x1)
    h = max(0.0, y2 - y1)
    cx = x1 + w / 2.0
    cy = y1 + h / 2.0
    # normalize
    return [cx / img_w, cy / img_h, w / img_w, h / img_h]

def save_yolo_label_txt(txt_path, dets, save_conf=True):
    """Save detections to YOLO .txt file.
    dets: list of dicts: {"cls": int, "xyxy": [x1,y1,x2,y2], "conf": float, "img_w": int, "img_h": int}
    """
    lines = []
    for d in dets:
        xywhn = xyxy_to_xywh_norm(d["xyxy"], d["img_w"], d["img_h"])
        if save_conf:
            lines.append(f"{d['cls']} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f} {d['conf']:.6f}")
        else:
            lines.append(f"{d['cls']} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}")
    Path(txt_path).parent.mkdir(parents=True, exist_ok=True)
    with open(txt_path, 'w') as f:
        f.write("\n".join(lines))

def draw_boxes(img_bgr, dets, class_names):
    out = img_bgr.copy()
    for d in dets:
        x1, y1, x2, y2 = map(int, d["xyxy"])
        label = f"{class_names[d['cls']]} {d['conf']:.2f}"
        # Draw rectangle and label
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
        ((tw, th), baseline) = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(out, (x1, y1 - th - baseline - 4), (x1 + tw + 4, y1), (0, 255, 0), -1)
        cv2.putText(out, label, (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
    return out

In [ ]:
import torch
print(torch.version.cuda)         # CUDA version PyTorch sees
print(torch.cuda.is_available())  # Should be True
print(torch.cuda.device_count())  # Should be >= 1


## 3) Load or train a model
- If `TRAIN_MODE='skip'`, we'll load `CUSTOM_MODEL_PATH` if provided, else a pre-trained YOLO model like `yolov8n.pt`.
- If `TRAIN_MODE='train'`, we'll fine-tune on `DATA_YAML` for `EPOCHS`.
- If `TRAIN_MODE='resume'`, we'll resume training from `CUSTOM_MODEL_PATH`.

In [4]:
assert TRAIN_MODE in {"skip", "train", "resume"}

if TRAIN_MODE == "skip":
    if CUSTOM_MODEL_PATH and Path(CUSTOM_MODEL_PATH).exists():
        print(f"Loading custom model from {CUSTOM_MODEL_PATH}")
        model = YOLO(CUSTOM_MODEL_PATH)
    else:
        print(f"Loading pretrained model: {PRETRAINED_MODEL_NAME}")
        model = YOLO(PRETRAINED_MODEL_NAME)
elif TRAIN_MODE == "train":
    assert Path(DATA_YAML).exists(), "DATA_YAML not found."
    model = YOLO(PRETRAINED_MODEL_NAME)
    res = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        # nudge losses (tune if needed)
        box=7.5,             # localization weight (default ~7.5)
        cls=1.0,             # classification weight (default ~0.5–1.0)
        dfl=1.5,             # distribution focal loss weight
        # a couple of stability helpers
        cos_lr=True,
        amp=True
    )
    print("Training complete.")
elif TRAIN_MODE == "resume":
    assert CUSTOM_MODEL_PATH and Path(CUSTOM_MODEL_PATH).exists(), "Provide CUSTOM_MODEL_PATH to resume."
    model = YOLO(CUSTOM_MODEL_PATH)
    res = model.train(resume=True, device=DEVICE)
    print("Resumed training.")

CLASS_NAMES = model.names
print("Model ready. Classes:", CLASS_NAMES)

Loading custom model from runs/detect/first/weights/best.pt
Model ready. Classes: {0: 'Caries', 1: 'Ulcer', 2: 'Tooth Discoloration', 3: 'Gingivitis'}


## 4) Batch predict and save YOLO labels
This cell:
1. Scans your `NEW_IMAGES_DIR` for images
2. Runs inference in batches
3. Writes YOLO-format labels to `NEW_IMAGES_DIR/labels/` with the same filename stem
4. writes preview images into `NEW_IMAGES_DIR/pred_vis/`

In [5]:
img_paths = [p for p in glob.glob(str(Path(NEW_IMAGES_DIR) / "**" / "*"), recursive=True)
             if Path(p).suffix.lower() in img_exts]
img_paths = sorted(img_paths)
print(f"Found {len(img_paths)} images in {NEW_IMAGES_DIR}")

batch_size = 16
n_batches = (len(img_paths) + batch_size - 1) // batch_size

saved = 0
for bi in tqdm(range(n_batches)):
    batch = img_paths[bi*batch_size : (bi+1)*batch_size]
    if not batch:
        continue
    results = model.predict(
        source=batch,
        conf=CONF_THRES,
        iou=IOU_THRES,
        max_det=MAX_DET,
        imgsz=IMGSZ,
        device=DEVICE,
        verbose=False
    )
    for img_path, r in zip(batch, results):
        img = cv2.imread(img_path)
        if img is None:
            print(f"[WARN] Could not read {img_path}")
            continue
        H, W = img.shape[:2]

        dets = []
        if r and hasattr(r, 'boxes') and r.boxes is not None:
            boxes = r.boxes.xyxy.cpu().numpy() if r.boxes.xyxy is not None else np.zeros((0,4))
            clss  = r.boxes.cls.cpu().numpy().astype(int) if r.boxes.cls is not None else np.zeros((0,), dtype=int)
            confs = r.boxes.conf.cpu().numpy() if r.boxes.conf is not None else np.zeros((0,))
            for b, c, cf in zip(boxes, clss, confs):
                dets.append({
                    "cls": int(c),
                    "xyxy": [float(b[0]), float(b[1]), float(b[2]), float(b[3])],
                    "conf": float(cf),
                    "img_w": W,
                    "img_h": H,
                })

        # Save YOLO label .txt with same stem under NEW_IMAGES_DIR/labels/
        stem = Path(img_path).stem
        txt_out = Path(OUTPUT_LABELS_DIR) / f"{stem}.txt"
        save_yolo_label_txt(txt_out, dets, save_conf=SAVE_CONF)
        saved += 1

        # Optional preview image with boxes
        # if SAVE_VIS and len(dets) > 0:
        vis = draw_boxes(img, dets, CLASS_NAMES)
        cv2.imwrite(str(Path(OUTPUT_VIS_DIR) / f"{stem}.jpg"), vis)

print(f"Wrote labels for {saved} images to: {OUTPUT_LABELS_DIR}")

Found 500 images in C:\Users\omer_\PycharmProjects\YOLO\new_images


100%|██████████| 32/32 [00:15<00:00,  2.04it/s]

Wrote labels for 500 images to: C:\Users\omer_\PycharmProjects\YOLO\new_images\labels


In [5]:
# Set the path to your dataset
images_dir = NEW_IMAGES_DIR
labels_dir = OUTPUT_LABELS_DIR

# Make sure labels directory exists
os.makedirs(labels_dir, exist_ok=True)

# Collect all image files
image_extensions = [".jpg", ".jpeg", ".png", ".bmp"]  # adjust if needed
image_files = [f for f in os.listdir(images_dir) if os.path.splitext(f)[1].lower() in image_extensions]

# Loop through and check for missing txt files
for img_file in image_files:
    base_name = os.path.splitext(img_file)[0]
    txt_path = os.path.join(labels_dir, base_name + ".txt")
    
    if not os.path.exists(txt_path):
        # Create an empty file
        open(txt_path, 'w').close()

print("✅ Blank txt files created for all images without annotations.")


✅ Blank txt files created for all images without annotations.


## 5) Move data
- Using to_move.txt, move good images and labels to the train folder.

In [7]:
def move_selected_images(
    base_dir: str,
):
    base_dir = Path(base_dir).resolve()
    txt_path = base_dir / "to_save.txt"
    new_images_dir = base_dir / "new_images"
    new_labels_dir = new_images_dir / "labels"
    pred_vis_dir = new_images_dir / "pred_vis"

    # YOLO train dirs
    train_images_dir = base_dir / "dataset" / "images" / "train"
    train_labels_dir = base_dir / "dataset" / "labels" / "train"
    train_images_dir.mkdir(parents=True, exist_ok=True)
    train_labels_dir.mkdir(parents=True, exist_ok=True)

    # Read names from to_save.txt
    if not txt_path.exists():
        print(f"[ERROR] {txt_path} not found.")
        return
    with open(txt_path, "r") as f:
        names_to_save = {line.strip() for line in f if line.strip()}

    if not names_to_save:
        print("[INFO] No names found in to_save.txt.")
        return

    # Prepare timestamp
    timestamp = datetime.now().strftime("%d_%m_%Y_%H_%M")

    moved_count = 0
    for name in sorted(names_to_save):
        # Find matching image file (case-insensitive)
        matches = list(new_images_dir.glob(f"{name}.*"))
        matches = [m for m in matches if m.is_file() and m.suffix.lower() in img_exts]
        if not matches:
            print(f"[WARN] No image found for '{name}' in {new_images_dir}")
            continue

        for img_path in matches:
            moved_count += 1
            new_basename = f"{timestamp}_{img_path.stem}{img_path.suffix.lower()}"
            new_labelname = f"{timestamp}_{img_path.stem}.txt"

            # Move and rename image
            shutil.move(str(img_path), train_images_dir / new_basename)

            # Move and rename label if exists
            label_path = new_labels_dir / f"{img_path.stem}.txt"
            if label_path.exists():
                shutil.move(str(label_path), train_labels_dir / new_labelname)
            else:
                print(f"[WARN] No label found for {img_path.stem}")

    # === Save a copy of to_save.txt to transfer_log before clearing ===
    transfer_log_dir = base_dir / "transfer_log"
    transfer_log_dir.mkdir(parents=True, exist_ok=True)
    log_file = transfer_log_dir / f"{timestamp}_LOG.txt"
    shutil.copy2(txt_path, log_file)

    # Clear remaining labels and pred_vis
    if new_labels_dir.exists():
        for f in new_labels_dir.glob("*.txt"):
            f.unlink()
    if pred_vis_dir.exists():
        for f in pred_vis_dir.glob("*"):
            if f.is_file():
                f.unlink()

    # Clear to_save.txt
    with open(txt_path, "w") as f:
        pass

    print(f"[DONE] Moved {moved_count} images+labels to {train_images_dir} / {train_labels_dir}.")

move_selected_images(BASE_DIR)


[DONE] Moved 257 images+labels to C:\Users\omer_\PycharmProjects\YOLO\dataset\images\train / C:\Users\omer_\PycharmProjects\YOLO\dataset\labels\train.
